In [1]:
!pip install kfp

In [2]:
from typing import Dict, List

import kfp
import kfp.dsl as dsl
from kfp import compiler
from kfp.dsl import Input, InputPath, Output, OutputPath, Dataset, Model, component
from kfp.dsl import component as component
from kfp import kubernetes
import time


/opt/conda/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/opt/conda/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


In [3]:
BASE_IMAGE = "traininghost/pipelineimage:latest"

In [4]:
@component(
    base_image=BASE_IMAGE, 
    packages_to_install=['scikit-learn', 'pandas', 'tensorflow']
)
def train_autoencoder_model(featurepath: str, epochs: str, modelname: str, modelversion:str):
    
    import time  
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import train_test_split


    # tensorflow library
    import tensorflow as tf
    from tensorflow.keras import layers, models
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.models import Model
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed
    from tensorflow.keras import regularizers, callbacks
    from tensorflow.keras.optimizers import Adam

    # Evaluation Indicators library
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
    from sklearn.metrics import precision_recall_curve, roc_curve, auc, confusion_matrix

    import requests
    import os
    from featurestoresdk.feature_store_sdk import FeatureStoreSdk
    from modelmetricsdk.model_metrics_sdk import ModelMetricsSdk

    mm_sdk = ModelMetricsSdk()
    fs_sdk = FeatureStoreSdk()
    print("featurepath is: ", featurepath)

    feature_list = [
        'phy_ul_n_samples', 'phy_dl_n_samples', 'mac_dl_buffer', 'mac_dl_brate', 'mac_ul_brate', 
        'mac_dl_ok', 'mac_dl_nok', 'mac_ul_ok', 'mac_ul_nok','timestamp', 'label_encoded'
    ]
    Data = fs_sdk.get_features(featurepath, feature_list)
    print("Dataframe:")
    print(Data)

    # benign data
    benign_data = Data[Data['label_encoded'] == 0]
    benign_data = benign_data.drop('label_encoded', axis=1)

    # attack data
    attack_data = Data[Data['label_encoded'] == 1]
    # attack data
    attack_data = Data[Data['label_encoded'] != 0]
    attack_data = attack_data.drop('label_encoded', axis=1)  # 이 줄을 꼭 추가해 주세요!

    # Split the benign into train, validation, test
    benign_train, benign_valtest = train_test_split(benign_data, test_size=0.4, random_state=42)
    benign_validation, benign_test = train_test_split(benign_valtest, test_size=0.5, random_state=42)

    # Check each dataset shape
    print("Train shape = ", benign_train.shape)
    print("Validation shape = ", benign_validation.shape)
    print("Test shape = ", benign_test.shape)

    from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
    from sklearn.metrics import precision_recall_curve, roc_curve, auc, confusion_matrix 
    attack_data = attack_data.iloc[:1500, :]
    print("\nattack data shape = ", attack_data.shape)

    # Check dataset feature
    print("benign test feature = ", benign_test.columns)
    print("attack feature = ", attack_data.columns)

    # Make test datasets
    test_data = np.concatenate([benign_test, attack_data], axis=0)

    # Make label of test datasets
    label = np.array(
        [0 for _ in range(len(benign_test))] +
        [1 for _ in range(len(attack_data))]
    )

    # ---------------------------
    # 0) 공격 데이터를 검증/테스트로 나눔 (라벨을 가진 검증 셋 확보)
    # ---------------------------
    # 기존: attack_data = attack_data.iloc[:1500, :]
    # 수정: 검증/테스트 분할
    attack_val, attack_test = train_test_split(attack_data, test_size=0.8, random_state=42)
    # (예: 20%만 검증에 사용. 필요시 비율 조정)

    print(f"attack_val shape = {attack_val.shape}, attack_test shape = {attack_test.shape}")

    # ---------------------------
    # 1) 테스트 셋 구성 (정상 테스트 + 공격 테스트)
    # ---------------------------
    test_data_np = np.concatenate([benign_test, attack_test], axis=0)
    test_label_np = np.array(
        [0 for _ in range(len(benign_test))] +
        [1 for _ in range(len(attack_test))])

    # 데이터프레임화 (기존 로직과 동일)
    test_data = pd.DataFrame(test_data_np)
    label = pd.DataFrame(test_label_np)

    all_test_data = pd.concat([test_data, label], axis=1)
    pre_features_list = ['phy_ul_n_samples', 'phy_dl_n_samples','mac_dl_buffer',
                         'mac_dl_brate', 'mac_ul_brate', 'mac_dl_ok', 'mac_dl_nok', 
                     'mac_ul_ok', 'mac_ul_nok', 'timestamp', 'label_encoded']
    all_test_data.columns = pre_features_list
    shuffled_test_data = all_test_data.sample(frac=1).reset_index(drop=True)

    features_list = ['phy_ul_n_samples', 'phy_dl_n_samples', 'mac_dl_buffer', 'mac_dl_brate', 
             'mac_ul_brate', 'mac_dl_ok', 'mac_dl_nok', 'mac_ul_ok', 'mac_ul_nok']

    benign_train = benign_train[features_list]
    benign_validation = benign_validation[features_list]
    test_data = shuffled_test_data[features_list]
    label = shuffled_test_data.label_encoded  # y_test

    # ---------------------------
    # 2) 검증 셋 구성 (정상 검증 + 공격 검증)  ← ROC–Youden을 위해 라벨 필요
    # ---------------------------
    val_df = pd.concat([
        pd.DataFrame(benign_validation).assign(label_encoded=0),
        pd.DataFrame(attack_val[features_list]).assign(label_encoded=1)
    ], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

    X_val_df = val_df[features_list]
    y_val = val_df['label_encoded'].values  # 검증 라벨

    # ---------------------------
    # 3) 스케일링
    # ---------------------------
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(benign_train)
    X_val   = scaler.transform(X_val_df)
    X_test  = scaler.transform(test_data)

    # ---------------------------
    # 4) AE 정의/학습 (기존과 동일)
    # ---------------------------
    input_dim = X_train.shape[1]
    autoencoder = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dense(16, activation='relu'),
        layers.Dense(8,  activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(input_dim, activation='linear')
    ])
    autoencoder.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005), loss='mse')
    history = autoencoder.fit(
        X_train, X_train,
        epochs=int(epochs), batch_size=64,
        validation_data=(X_val, X_val),
        shuffle=True, verbose=1
    )

    # ---------------------------
    # 5) 검증 셋에서 재구성 오차 계산 → ROC–Youden 임계값
    # ---------------------------
    t0 = time.perf_counter()
    X_val_pred = autoencoder.predict(X_val, verbose=0)
    elapsed_val = time.perf_counter() - t0
    latency_val_ms = (elapsed_val / len(X_val)) * 1000.0
    throughput_val_rps = len(X_val) / elapsed_val if elapsed_val > 0 else float('inf')
    print(f"[Predict Time] Validation set: total {elapsed_val:.4f}s, per-sample {latency_val_ms:.4f} ms, "
        f"throughput {throughput_val_rps:.1f} samples/s")
    val_mse = np.mean(np.power(X_val - X_val_pred, 2), axis=1)

    fpr, tpr, thr = roc_curve(y_val, val_mse)           # 연속값(val_mse)로 ROC
    youden = tpr - fpr
    best_idx = np.argmax(youden)
    tau = thr[best_idx]
    print(f"[Threshold | ROC–Youden] tau = {tau:.6f} (idx={best_idx}, TPR={tpr[best_idx]:.3f}, FPR={fpr[best_idx]:.3f})")

    # ---------------------------
    # 6) 테스트 셋 평가 (임계값 고정)
    # ---------------------------
    t0 = time.perf_counter()
    X_test_pred = autoencoder.predict(X_test, verbose=0)
    elapsed_test = time.perf_counter() - t0
    latency_test_ms = (elapsed_test / len(X_test)) * 1000.0
    throughput_test_rps = len(X_test) / elapsed_test if elapsed_test > 0 else float('inf')
    print(f"[Predict Time] Test set: total {elapsed_test:.4f}s, per-sample {latency_test_ms:.4f} ms, "
        f"throughput {throughput_test_rps:.1f} samples/s")

    test_mse = np.mean(np.power(X_test - X_test_pred, 2), axis=1)

    y_pred = (test_mse > tau).astype(int)
    y_true = np.array(label)


    print("\n[Evaluation Results]")
    
    acc = accuracy_score(y_true, y_pred)
    auc_val = roc_auc_score(y_true, test_mse)
    f1 = f1_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)

    print(f"Accuracy  : {acc:.4f}")
    print(f"AUC(score): {auc_val:.4f}")
    print(f"F1        : {f1:.4f}")
    print(f"Precision : {pre:.4f}")
    print(f"Recall    : {rec:.4f}")
    
    import json
    data = {}
    data['metrics'] = []
    data['metrics'].append({
        'Accuracy': str(acc),
        'F1-Score': str(f1),
        'Precision': str(pre),
        'Recall': str(rec)
    })
    
    artifactversion = "1.0.0"
    mm_sdk.upload_metrics(data, 1)
    
    export_dir = "/tmp/autoencoder_model"
    export_root = "/tmp/autoencoder_model"
    
    autoencoder.export(export_dir)
    mm_sdk.upload_model(str(export_root), modelname, modelversion, artifactversion)


In [5]:
@dsl.pipeline(
    name='Autoencoder_Pipeline',
    description='Anomaly Detection Using Autoencoder ver2.'
)
def autoencoder_pipeline(
    featurepath: str, epochs: str, modelname: str, modelversion:str):
    
    trainop=train_autoencoder_model(featurepath=featurepath, epochs=epochs, modelname=modelname, modelversion=modelversion)
    trainop.set_caching_options(False)
    kubernetes.set_image_pull_policy(trainop, "IfNotPresent")

In [6]:
pipeline_func = autoencoder_pipeline
file_name = "Autoenocoder_Pipeline"
kfp.compiler.Compiler().compile(pipeline_func, f'{file_name}.yaml')

pipeline_func = autoencoder_pipeline
file_name = "Autoencoder_pipeline"

kfp.compiler.Compiler().compile(pipeline_func, '{}.yaml'.format(file_name))

# YAML 파일 업로드
import requests
pipeline_name = "Autoencoder_Pipeline"
pipeline_file = file_name+'.yaml'
requests.post(f"http://tm.traininghost:32002/pipelines/{pipeline_name}/upload", files={'file': open(f"{file_name}.yaml", 'rb')})

<Response [200]>